# ☕🤖 RoboBarista – 3 Kameras → FAIRINO NewSpline

**Dies ist der neue Standardablauf.** Wir zeichnen eine virtuelle Kannenbewegung mit drei RGB-D-Kameras und AprilTags auf und übergeben ihre 6D-Wegpunkte danach an die offiziellen FAIRINO-Bewegungsfunktionen `NewSpline` bzw. `MoveL` mit Blend.

**Kein eigener Ruckig- oder Drehmomentregler mehr in dieser Notebook-Pipeline.**

Dieses Notebook ist ein **Offline-Trockenlauf**: Kein echter FAIRINO-Controller wird verbunden. Wir erzeugen gültige SDK-Befehle, aber keine von der Herstellerfirmware berechnete Roboterbewegung.

Die Aufnahme ist eine virtuelle, aus dem MuJoCo-Modell generierte Demonstration – keine echte menschliche Latte-Art-Aufzeichnung.


In [ ]:
# Colab: CPU genügt; die ersten virtuellen RGB-D-Render können dauern.
import os,subprocess,json
from pathlib import Path
if not Path("/content/robocasa").exists():
    subprocess.run(["git","clone","--depth","1","--branch","feature/ridxm-real-latte-trajectory-2026-10-23","https://github.com/Coffee-Tobi/robocasa.git",
                    "/content/robocasa"],check=True)
os.chdir("/content/robocasa")
subprocess.run(["apt-get","update","-qq"],check=True)
subprocess.run(["apt-get","install","-y","-qq","libosmesa6-dev","libgl1","ffmpeg"],check=True)
subprocess.run(["python","-m","pip","install","-q","-r","requirements.txt"],check=True)
env={**os.environ,"MUJOCO_GL":"osmesa","PYOPENGL_PLATFORM":"osmesa"}
print("RoboBarista Integration-Branch vorbereitet (nicht main).")
print("FAIRINO/Mujoco = virtuell, keine reale Hardware-Verbindung.")

## Schritt 1 – Mit drei RGB-D-Kameras AprilTags aufnehmen

Der virtuelle Roboter generiert eine Demonstration. **Nur die drei gerenderten Kamerabilder** werden zur 6D-Poseberechnung verwendet; der Quellweg dient nur zur nachträglichen Auswertung.

Mit `--capture-only` unterbleibt der alte IK-/Ruckig-Nachfahrversuch.

In [ ]:
subprocess.run(["python","-u","-m","robobarista.camera_joint_replay",
    "--capture-only","--seed","14","--frames","31","--fps","15",
    "--width","960","--height","720","--output","runs/fairino_camera_capture"],
    check=True,env=env)
camera=json.loads(Path("runs/fairino_camera_capture/camera_measurements.json").read_text())
print("AprilTag-Frames:",len(camera),"gültig:",sum(x["valid"] for x in camera))
print("Kameras sichtbar:",sorted({x["visible_views"] for x in camera}))
# NEU: Exportiert aus denselben AprilTag-Messungen einen portablen
# Zwischendatensatz. Die Original-Posen liegen nur separat zur Bewertung.
from robobarista.latte_dataset_bridge import export_camera_capture
export_report=export_camera_capture(
    "runs/fairino_camera_capture","runs/apriltag_dataset_export")
print("Trainingsdaten-Export:",json.dumps(export_report,indent=2))

## Schritt 2 – FAIRINO NewSpline aus diesen Kamerapunkten vorbereiten

Die Demonstration wird für unsere aktuelle Übung auf **18 mm Kannenmittelpunkt-Auslenkung**, maximal 20 mm, skaliert und zeitlich auf 20 Sekunden gestreckt. Das ist **nicht** der ursprüngliche unveränderte Bewegungsverlauf. Zusätzlich wird der Ausgießerweg mit einem **nur für das bisherige virtuelle Kannenmodell bekannten** lokalen Offset überprüft; dieser ersetzt keine echte Werkzeugkalibrierung.

Wir übergeben an `NewSpline` jeden zweiten gemessenen 6D-Punkt (16 Wegpunkte) und geben die offizielle Befehlsfolge aus. Die Identitätskalibrierung gilt **nur im Offline-Test**.

In [ ]:
subprocess.run(["python","-u","-m","robobarista.fairino_barista_pipeline",
    "--input","runs/fairino_camera_capture/camera_measurements.json",
    "--output","runs/fairino_standard","--target-mm","18",
    "--duration-s","20","--stride","2",
    "--virtual-spout-offset-m","-.073","0",".117"],check=True)
summary=json.loads(Path("runs/fairino_standard/summary.json").read_text())
print(json.dumps(summary,indent=2))

## Was ist geschafft – und was noch nicht?

- **Geschafft:** drei Kameraansichten, AprilTag-Erkennung, 6D-Aufnahme, FAIRINO-Wegpunktbefehle.
- **Nicht geschafft:** einen echten FAIRINO-Controller bzw. dessen NewSpline-Interpolation starten, reale Greiferkräfte und Flüssigkeitsphysik messen oder echte Roboterbewegungen nachweisen.
- **Für Hardware noch erforderlich:** Kamera-zu-Roboter-Kalibrierung, Greifer-/Kannengriff-Versatz, Kollisionsfreigabe und explizite Bewegungserlaubnis.

Im Ordner `runs/fairino_standard` findest du die erzeugten Dateien.

In [ ]:
from IPython.display import display
from google.colab import files
from pathlib import Path
for name in ("fairino_new_spline_commands.json","summary.json"):
    p=Path("runs/fairino_standard")/name
    print(name,":",p.stat().st_size,"Bytes")
print("Video der Kameraaufnahme:",Path("runs/fairino_camera_capture/camera_capture.mp4").exists())
print("Diese Befehle sind OFFLINE-DRY-RUN, keine reale FAIRINO-Roboterbewegung.")

## Zusätzlich: kompletter Espresso- und Milchablauf mit zwei Greifern

Dies ist unser neuer 34,5-Sekunden-Film mit elf Schritten (Siebträger, Mahlen, Tamper, Brühgruppe, Becher, Espresso, Milchdosierung, Dampfdüse, Gießen und Reinigen). Die Sensoranzeigen sind SYNTHETISCH, nicht aus echten Kontakten. Die Roboterarmbewegungen zeigen weiterhin sichere Vorpositionen.

In [ ]:
# Dies erzeugt zusätzlich den kompletten 11-Schritte-Film mit Sensor-Konzeptgrafiken.
from IPython.display import Video, display
subprocess.run(["python","-u","-m","robobarista.full_barista_sensor_video",
    "--output","runs/whole_coffee_with_FT","--frames-leg","9",
    "--capture-fps","6","--repeat","2","--width","640","--height","480"],
    check=True,env=env)
display(Video("runs/whole_coffee_with_FT/complete_barista_with_sensor_concepts.mp4",
              embed=True))
print("Hinweis: vorgeprüfte Vorpositionen; Druck-/Drehmomentwerte synthetisch.")

## Hardware-Reife und wiederverwendete Open-Source-Funktionen

Unsere Kaffee-Station verwendet die **MIT-lizenzierte RoboCasa-Kaffeemaschinenlogik** zur Kontrolle, ob der Becher unter dem Espressoauslauf steht. Unpassende STEP/STL-Downloads werden automatisch als leere CAD-Daten abgelehnt.

Die neue FAIRINO-Prüfung kann später die **8 herstellerseitigen IK-Lösungen**, die tatsächliche TCP-Pose, Gelenkgeschwindigkeit, Not-Aus- und Kraftsensorwerte auslesen. Hier werden keinerlei Hardwarebefehle gesendet. Unser reiner Softwaretest ist kein Sicherheitsnachweis.

In [ ]:
from robobarista.robocasa_coffee_adapter import CoffeeMachineState
from robobarista.fairino_sdk_preflight import hardware_readiness_gate
print("Symbolische Espresso-Station:",CoffeeMachineState().demo_brew_step((.440,.205,.882)))
gates=hardware_readiness_gate()
print("Für autonomes echtes Latte Art freigegeben:",gates["hardware_ready_for_autonomous_espresso_latte_art"])
print("Fehlende Hardwaretests:",", ".join(gates["missing_evidence"]))
print("Recherche und FAIRINO-Methode: planning/OSS_BARISTA_FR5_READINESS_AUDIT.md")

## Neuer realistischer AprilTag-Test (optional)

Der normale Kamera-Test weiter oben benutzt die **großen historischen Tags**. Dieses Experiment baut eine andere, mechanisch plausiblere Szene mit **36/32/30 mm** Kannen-Tags, **28 mm** am Becher, **24 mm** am Siebträgergriff und **36–42 mm** an den Geräten. Statt hochstehenden 10-cm-Tafeln sind die Tags mit dünnen, starren Halterungen oder flach auf einer festen Oberfläche montiert.

**Wichtig:** Die 3 Kameras müssen näher stehen. Der erste erfolgreiche Test zeigte 13/13 gültige Zwei-Kamera-Bewegungen, aber nur 2/8 erkannte stationäre Markierungen. Die folgende Zelle führt einen neuen Test durch und überprüft, welche Referenzen jetzt wirklich sichtbar sind.

In [ ]:
import os,subprocess,json
from pathlib import Path
from IPython.display import display,Video
env={**os.environ,"MUJOCO_GL":"osmesa","PYOPENGL_PLATFORM":"osmesa"}
subprocess.run(["python","-u","-m","robobarista.camera_joint_replay",
    "--capture-only","--marker-profile","compact",
    "--frames","13","--fps","5","--seed","14",
    "--width","1280","--height","960",
    "--output","runs/compact_tags"],env=env,check=True)
p=Path("runs/compact_tags")
summary=json.loads((p/"summary.json").read_text())
print("Zwei-Kamera-Pose gültig:",summary["camera_two_view_frames"],"/",summary["frames"])
print("Max. virtueller Trackingfehler:",summary["max_camera_vs_truth_mm"],"mm")
print("Stationen mit sichtbaren Markern:",summary["fixed_station_tags_recognized_in_initial_render"],"/",summary["fixed_station_reference_count"])
display(Video(str(p/"camera_capture.mp4"),embed=True))

### OPTIONAL: Echte Hugging Face LeRobot-Bibliothek verwenden

Unser [LeRobot-Dataset-Adapter](https://github.com/Coffee-Tobi/robocasa/blob/feature/compact-realistic-apriltag-fixtures/robobarista/lerobot_native_latte.py) benutzt tatsächlich `LeRobotDataset.create()`, `add_frame()`, `save_episode()` und `finalize()`. Dazu müssen die **drei unbearbeiteten Kamerabilder** gespeichert sein.

Der Adapter bildet vorerst nur eine **synthetische Kannenbewegung** ab, nicht die menschlichen Demonstrationen des fremden Projekts und auch keine FAIRINO-Gelenkaktionen. Für einen echten Roboter sollten wir die 6 Gelenkwinkel und den Greiferstatus synchron ergänzen.

In [ ]:
# Installiert LeRobot nur bei Bedarf; kann große Abhängigkeiten nachziehen.
# !pip install "lerobot>=0.6"
#
# Danach mit --save-camera-frames erneut aufnehmen:
# !python -m robobarista.camera_joint_replay --capture-only \
#   --marker-profile compact --save-camera-frames --frames 13 --fps 5 \
#   --width 1280 --height 960 --output runs/compact_lerobot
# !python -m robobarista.lerobot_native_latte \
#   --capture runs/compact_lerobot --output runs/lerobot_virtual_latte
print("LeRobot-Modul ist optional; keine echte menschliche Trainingsaufnahme vorhanden.")

## NEU: Qualitätsprüfung, echte DFKI DMP, MoveIt-Export & FAIRINO-Readiness

Die folgenden Zellen sind **optionale Offline-Versuche** auf dem Integrationsbranch. `main` wurde nicht verändert.

- Kamera-Qualitätsbericht prüft virtuelle 0,5-mm-Positionsgrenze und 8/8 Stations-Referenzen; ein erfolgreicher Render bedeutet nicht die Zielpräzision.
- DFKI `movement_primitives` lernt die virtuelle Spout-XYZ-Bahn aus AprilTag-Messungen und kann das Endziel begrenzt verschieben; **keine Orientierung, keine echte Milch und keine Robotersteuerung**.
- MoveIt2-Szenenexport und SimMachine-Readiness sind nur Vorbereitung. Ein ROS2-MoveIt-/FAIRINO-Controller läuft nicht in dieser Colab-Zelle.



In [ ]:
from pathlib import Path
from robobarista.vision_quality import evaluate_camera_quality
import json
compact_dir=Path("runs/compact_tags")
if (compact_dir/"summary.json").exists() and (compact_dir/"camera_vs_truth_EVAL_ONLY.json").exists():
    quality=evaluate_camera_quality(
        json.loads((compact_dir/"summary.json").read_text()),
        json.loads((compact_dir/"camera_vs_truth_EVAL_ONLY.json").read_text()))
    print("Virtuelles Kannenpositionsziel (< = 0,5mm):",quality["virtual_position_target_met"])
    print("Station-Referenzen vollständig:",quality["station_coverage_target_met"])
    print("Erkannte stationäre Tags:",quality["visible_reference_stations"],"/",quality["total_reference_stations"])
    print("Maximale simulierte Kannenpositionsabweichung (mm):",quality["max_virtual_pitcher_position_error_mm"])
else:
    print("Bitte zuerst die optionale Zelle 'compact_tags_vs_old' zur kleinen Markeraufnahme ausführen.")


In [ ]:
# Optionaler externer DFKI-DMP-Adapter, NUR offline und kein Roboterbefehl.
subprocess.run(["python","-m","pip","install","-q","movement_primitives>=0.9,<1"],check=True)
camera_path=Path("runs/fairino_camera_capture/camera_measurements.json")
assert camera_path.exists(), "Bitte zuerst die 3-Kamera-Aufnahme oben ausführen"
subprocess.run(["python","-m","robobarista.latte_dmp",
    "--camera",str(camera_path),"--output","runs/latte_dmp/offline_tip_path.json",
    "--virtual-tip-offset-m","-.069","0",".136","--goal-offset-mm","1","0","0"],
    check=True,env=env)
dmp=json.loads(Path("runs/latte_dmp/offline_tip_path.json").read_text())
print("Gelernter Offline-Spout-DMP:",dmp["status"])
print("Auslenkung Ausgießerspitze (virtuell, mm):",round(dmp["predicted_tip_excursion_mm"],3))
print("Reale Roboteransteuerung erlaubt:",dmp["hardware_execution_allowed"])


In [ ]:
from robobarista.moveit_scene_bridge import export_scene
from robobarista.vendor_runtime_preflight import inspect_runtime
scene=export_scene()
Path("runs/moveit2").mkdir(parents=True,exist_ok=True)
Path("runs/moveit2/nominal_collision_scene.json").write_text(json.dumps(scene,indent=2))
print("MoveIt2-Kollisionsobjekte exportiert:",len(scene["objects"]))
runtime=inspect_runtime()
print("ROS2 MoveIt-Umgebung verfügbar:",runtime["fairino_ros2_humble_ready_to_attempt_moveit_demo"])
print("FAIRINO SimMachine tatsächlich verbunden:",runtime["simmachine_started_and_reachable"])
print("Diese Zelle führt KEINE Roboterbewegungen aus.")


### Optionaler Vergleich: Latte-Art-Quelle → Kameramessung → bewegter FR5

Die folgende Zelle rendert eine neue synthetische 20-Sekunden-Demonstration mit 61 Kamera-Messbildern und erstellt anschließend drei zeitlich vergleichbare Videos. Das braucht auf Colab-CPU viele gerenderte Bilder. Sie ist daher standardmäßig deaktiviert. Setze `RUN_THREE_VIDEOS=True`, um sie explizit auszuführen. Das Nachfahren basiert auf idealisierter MuJoCo-IK, **nicht** auf FAIRINO-SimMachine oder realer Milch.



In [ ]:
RUN_THREE_VIDEOS=False
if RUN_THREE_VIDEOS:
    subprocess.run(["python","-u","-m","robobarista.camera_joint_replay",
        "--capture-only","--latte-art-source","--frames","61","--fps","3",
        "--width","960","--height","720","--output","runs/latte_capture"],
        env=env,check=True)
    subprocess.run(["python","-u","-m","robobarista.latte_art_three_videos",
        "--capture","runs/latte_capture","--output","runs/latte_three","--fps","8"],
        env=env,check=True)
    from IPython.display import Video,display
    for movie in ("original_20s.mp4","tracking_20s.mp4","robot_20s.mp4"):
        display(Video(str(Path("runs/latte_three")/movie),embed=True))
else:
    print("Optionale drei-Videos-Simulation nicht gestartet. RUN_THREE_VIDEOS=True setzen.")


## NEU – Echte Latte-Art-Gießbewegungen als Quelle (OPTIONAL)

Das öffentliche [ridxm/latte-pour-demos](https://huggingface.co/datasets/ridxm/latte-pour-demos)-Material enthält laut direkt geprüfter Metadaten 40 Demos mit **20 Hz, 14D Gelenk-/Greiferzuständen und 3 RGB-Kameras**. Die 14 Zustände sind sechs Gelenke links, linker Greifer, sechs Gelenke rechts, rechter Greifer. Keiner dieser Werte ist automatisch eine 6D-Position der Milchkanne.

Unsere Software trennt drei Schritte: (1) echtes LeRobot-Datenset lesen, (2) RM65-URDF-Vorwärtskinematik mit korrektem Greifermount, (3) eine unabhängige **neue virtuelle** AprilTag-Kameramessung und kinematischen FR5-Nachfahrversuch. Die drei Originalvideos besitzen keine von uns belegte metrische Kamerakalibrierung, und das Projekt simuliert die Milch nicht physikalisch. Die Original-Daten bleiben lokal und werden nicht ins GitHub-Repo hochgeladen.

Die nächsten Zellen sind bewusst deaktiviert, bis du ausdrücklich auf `True` stellst. Mit vorhandener Internetverbindung in Colab können wir eine Episode laden. Ein kompletter π0-Trainingslauf ist dafür nicht erforderlich.



In [ ]:
RUN_REAL_LATTE_IMPORT=False
REAL_EPISODE=0
if RUN_REAL_LATTE_IMPORT:
    import subprocess,json,os
    from pathlib import Path
    subprocess.run(["python","-m","pip","install","-q","huggingface_hub","pyarrow"],check=True)
    subprocess.run(["python","-m","robobarista.real_latte_dataset",
      "--parquet-only","--episode",str(REAL_EPISODE),
      "--output",f"runs/ridxm_episode_{REAL_EPISODE}"],
      check=True,env=env)
    report=json.loads(Path(f"runs/ridxm_episode_{REAL_EPISODE}/source_and_validity.json").read_text())
    print("Echte Demonstrationsframes:",report["frames"])
    print("Gemessene Frequenz:",report["median_hz"])
    print("Original 3D-Kannenposition bereits vorhanden?", report["flange_and_pitcher_pose_available"])
else:
    print("Datensatz-Import optional: RUN_REAL_LATTE_IMPORT=True zum Aktivieren")


### RM65-Geometrie und erneute Kameraaufnahme

Bevor die reale Gelenkbewegung auf unseren FR5 übertragen werden kann, benötigen wir ein **zur Originalhardware passendes** RealMan-RM65-URDF, die sechs Gelenknamen, den von der Person gehaltenen Arm sowie den starren Versatz zwischen Roboterflansch und Milchkannen-Marker. Das offizielle Modell ist über [RealMan rm_models](https://github.com/RealManRobot/rm_models) erhältlich.

Fehlt die Griff-/Flanschkalibrierung, gibt es einen ausdrücklich unkalibrierten Identitäts-Test. Dieser demonstriert nur geometrische Machbarkeit, **nicht** die exakte Kannenbewegung. Sollte ein 20-Sekunden-Datensatz auch das Aufnehmen/Transportieren der Kanne enthalten, muss vor der Latte-Art-Kameramessung der reine Gießabschnitt ausgewählt werden; die Software verweigert sonst zu große Bewegungen.


Der offizielle RealMan-Download enthält z. B. `RM65-B/urdf/RM65-B.urdf` mit `joint_1 … joint_6` und `link_6`. Das Colab kann diese Datei sparsam herunterladen. **Welcher RM65-Typ und welcher Arm tatsächlich aufgenommen wurden, ist damit noch nicht bewiesen.** Der Beispielschnitt 8–12 Sekunden muss anhand des Originalvideos auf die echte reine Gießphase geprüft werden.


In [ ]:
RUN_RM65_FK_AND_NEW_CAMERAS=False
RM65_URDF="/content/rm_models/RM65/urdf/RM65-B/urdf/RM65-B.urdf"  # durch verifiziertes offizielles URDF ersetzen
RM65_FLANGE_LINK="link_6"                      # aus URDF ablesen!
RM65_JOINTS=["joint_1","joint_2","joint_3","joint_4","joint_5","joint_6"] # überprüfen!
RECORDED_ARM="right"                         # erst Originalvideo überprüfen!
RECORDED_UNIT="rad"                          # Roh-JointState gegen Aufzeichnung prüfen!
UNCALIBRATED_IDENTITY_TOOL_ONLY=True
if RUN_RM65_FK_AND_NEW_CAMERAS:
    import subprocess
    from pathlib import Path
    source=Path(f"runs/ridxm_episode_{REAL_EPISODE}/recorded_realman_joints.json")
    assert source.exists(), "Originale Gelenkdaten zuerst importieren"
    if not Path(RM65_URDF).exists():
        subprocess.run(["git","clone","--depth","1","--filter=blob:none","--sparse",
          "https://github.com/RealManRobot/rm_models.git","/content/rm_models"],check=True)
        subprocess.run(["git","-C","/content/rm_models","sparse-checkout","set","RM65"],check=True)
    assert Path(RM65_URDF).exists(), "Offizielle Modellvariante RM65-B nicht vorhanden"
    print("WARNUNG: RM65-B ist nur eine OFFIZIELLE VERMUTETE Variante; nicht nachgewiesen als Original-Roboter.")
    args=["python","-m","robobarista.realman_fk_bridge",
          "--input-joints",str(source),"--urdf",RM65_URDF,
          "--flange",RM65_FLANGE_LINK,"--joint-names",*RM65_JOINTS,
          "--joint-unit",RECORDED_UNIT,"--arm",RECORDED_ARM,
          "--start-s","8","--end-s","12",
          "--output","runs/ridxm_realman_fk"]
    if UNCALIBRATED_IDENTITY_TOOL_ONLY:
        args.append("--assume-identity-tool")
        print("WARNUNG: Nur vorläufiger FK-Formtest, echte Kanne NICHT kalibriert")
    else:
        raise RuntimeError("Bitte zuerst gemessenes --mount-json für T_flange_pitcher konfigurieren")
    subprocess.run(args,check=True,env=env)
    subprocess.run(["python","-u","-m","robobarista.camera_joint_replay",
      "--capture-only","--marker-profile","compact","--camera-count","3",
      "--external-pitcher-trajectory","runs/ridxm_realman_fk/source_pitcher_6d_URDF_FK.json",
      "--frames","61","--fps","3","--width","1280","--height","960",
      "--output","runs/ridxm_remasured_three_cameras"],
      check=True,env=env)
else:
    print("RM65 FK + neue 3D-Kameramessung optional, benötigt Originaldaten und URDF")
